# mCCI-S: development and external validation
**Modified Charlson Comorbidity Index for Surgical ICU patients (mCCI-S)**
Development in MIMIC-IV v3.1, external validation in eICU-CRD v2.0 (185 hospitals).

Requirements: PhysioNet credentialed access to MIMIC-IV and eICU-CRD on Google BigQuery, and the analytic
tables created by the SQL scripts in `sql/` (01, 02 and 03) in your own BigQuery project.

Run in Google Colab: *Runtime > Run all*. The notebook writes **aggregate results only** (tables, model
coefficients, figures) to `mccis_results.zip`; no patient-level data leave BigQuery.

In [ ]:
SYNTHETIC = False            # True = run on simulated data with the same structure (for testing only)
PROJECT = "YOUR_GCP_PROJECT"  # your BigQuery project ID (dataset `mccis` created by the SQL scripts)
SEED = 2026
N_BOOT = 200                 # bootstrap resamples for internal validation
OUTDIR = "mccis_results"
POINT_UNIT = 0.25            # 1 point = 0.25 on the log-odds scale (OR ~1.28)

In [ ]:
import os, json, zipfile, warnings
import numpy as np, pandas as pd
import scipy.stats as st
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)
os.makedirs(OUTDIR, exist_ok=True)
expit = lambda x: 1 / (1 + np.exp(-x))
logit = lambda p: np.log(p / (1 - p))

## 1. Load cohorts
MIMIC-IV: first postoperative ICU stay per patient, non-cardiac, excluding neurological step-down units.
eICU-CRD: first operative ICU admission per patient, non-cardiac, surgical urgency recorded.

In [ ]:
def load_bigquery():
    from google.colab import auth
    auth.authenticate_user()
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT)
    q_m = f"""
      SELECT t.*, s.* EXCEPT (stay_id), a.deathtime AS adm_deathtime
      FROM `{PROJECT}.mccis.mimic_stays` AS t
      JOIN `{PROJECT}.mccis.mimic_scores` AS s USING (stay_id)
      LEFT JOIN `physionet-data.mimiciv_3_1_hosp.admissions` AS a ON a.hadm_id = t.hadm_id
      WHERE t.in_cohort = 1
        AND t.first_careunit NOT IN ('Neuro Intermediate', 'Neuro Stepdown')"""
    q_e = f"""
      SELECT * FROM `{PROJECT}.mccis.eicu_cohort`
      WHERE in_cohort = 1 AND emergency IS NOT NULL"""
    return client.query(q_m).to_dataframe(), client.query(q_e).to_dataframe()


def make_synthetic():
    """Testing only: random data with the same column structure as the BigQuery tables."""
    comps = ["mi", "chf", "pvd", "cvd", "dementia", "cpd", "rheum", "pud", "mild_liver", "severe_liver",
             "dm_nocc", "dm_cc", "hemi", "renal", "renal_sev", "malig_any", "heme", "solid", "mets", "aids"]
    prev = dict(zip(comps, [.1, .15, .12, .2, .03, .2, .04, .03, .1, .05, .18, .07, .06, .14, .05, .22, .02, .2, .1, .003]))

    def base(n, emerg_p):
        d = pd.DataFrame({c: rng.binomial(1, prev[c], n) for c in comps})
        d["age"] = np.clip(rng.normal(63, 16, n).round(), 18, 91).astype(int)
        d["emergency"] = rng.binomial(1, emerg_p, n)
        d["acute_cvd"] = rng.binomial(1, .1, n); d["tbi"] = rng.binomial(1, .08, n)
        lp = -5 + 0.03 * (d.age - 60) + 1.6 * d.emergency + 0.5 * d.chf + 0.6 * d.renal + 0.7 * d.mets + 0.4 * d.cvd
        d["icu_death"] = rng.binomial(1, expit(lp)); d["hosp_death"] = np.maximum(d.icu_death, rng.binomial(1, .02, n))
        return d, lp

    m, _ = base(10214, .69)
    m["surg_cat"] = rng.choice(["Abdominal/GU", "Neurosurgery", "Thoracic", "Orthopaedic", "Trauma", "Vascular",
                                "Head-neck/plastic", "Other"], len(m))
    m["sofa_d1"] = np.clip((rng.poisson(3, len(m)) + 3 * m.icu_death), 0, 20)
    m["emerg_strict"] = m.emergency * rng.binomial(1, .9, len(m)); m["n_prior_adm"] = rng.poisson(.7, len(m))
    for c in comps:
        m[f"p_{c}"] = m[c] * (m.n_prior_adm > 0) * rng.binomial(1, .6, len(m))
    m["intime"] = pd.Timestamp("2150-01-01") + pd.to_timedelta(rng.integers(0, 10 ** 6, len(m)), unit="m")
    m["outtime"] = m.intime + pd.to_timedelta(rng.integers(600, 9000, len(m)), unit="m")
    m["adm_deathtime"] = pd.NaT
    dd = m.icu_death == 1
    m.loc[dd, "adm_deathtime"] = m.loc[dd, "outtime"] + pd.to_timedelta(rng.choice([-60, -5, 120], dd.sum()), unit="m")
    e, lpe = base(16925, .15)
    e["hemi"] = rng.binomial(1, .002, len(e)); e["dm_cc"] = 0
    for c in comps:
        e[f"ph_{c}"] = e[c]; e[f"icd_{c}"] = e[c]
    e["acute_brain_dx"] = rng.binomial(1, .06, len(e))
    e["op_system"] = rng.choice(["Gastrointestinal", "Neurologic", "Respiratory", "Musculoskeletal/Skin", "Trauma",
                                 "Cardiovascular", "Genitourinary", None], len(e))
    e["hospitalid"] = rng.integers(1, 186, len(e))
    e["apache_pred_hosp"] = np.clip(expit(lpe + rng.normal(0, .8, len(e)) + 1), 0.001, 0.99)
    return m, e


mimic_raw, eicu_raw = make_synthetic() if SYNTHETIC else load_bigquery()
print("MIMIC-IV:", mimic_raw.shape, " eICU-CRD:", eicu_raw.shape)

## 2. Harmonised predictor definitions
- Hemiplegia and cerebrovascular disease are counted as *pre-existing* only (not when coded together with an
  acute stroke or traumatic brain injury in the same admission; MIMIC-IV also uses previous admissions).
- Diabetes is a single variable (eICU-CRD does not distinguish complicated diabetes).
- Solid tumour split into non-metastatic and metastatic; haematological malignancy separate.

In [ ]:
SURG_MAP_MIMIC = {"Neurosurgery": "neuro", "Thoracic": "thoracic", "Abdominal/GU": "abdominal_gu",
                  "Orthopaedic": "msk_trauma", "Trauma": "msk_trauma"}
SURG_MAP_EICU = {"Neurologic": "neuro", "Respiratory": "thoracic", "Gastrointestinal": "abdominal_gu",
                 "Genitourinary": "abdominal_gu", "Metabolic/Endocrine": "abdominal_gu", "Transplant": "abdominal_gu",
                 "Musculoskeletal/Skin": "msk_trauma", "Trauma": "msk_trauma"}


def harmonize(d, src):
    o = pd.DataFrame(index=d.index)
    o["age"] = d.age.astype(float)
    o["emergency"] = d.emergency.astype(int)
    o["icu_death"] = d.icu_death.astype(int)
    o["hosp_death"] = d.hosp_death.astype(int)
    for c in ["mi", "chf", "pvd", "dementia", "cpd", "rheum", "pud", "heme", "mets", "renal"]:
        o[c] = d[c].astype(int)
    o["dm"] = np.maximum(d.dm_nocc, d.dm_cc).astype(int)
    o["liver_sev"] = d.severe_liver.astype(int)
    o["liver_mild"] = ((d.mild_liver == 1) & (d.severe_liver == 0)).astype(int)
    o["solid_nonmet"] = ((d.solid == 1) & (d.mets == 0)).astype(int)
    if src == "mimic":
        acute = (d.acute_cvd == 1) | (d.tbi == 1)
        o["hemi_c"] = (((d.hemi == 1) & ~acute) | (d.p_hemi == 1)).astype(int)
        o["cvd_c"] = (((d.cvd == 1) & (d.acute_cvd == 0)) | (d.p_cvd == 1)).astype(int)
        o["surg_dom"] = d.surg_cat.map(SURG_MAP_MIMIC).fillna("other")
        o["sofa"] = d.sofa_d1.astype(float)
        o["emerg_strict"] = d.emerg_strict.astype(int)
        o["n_prior_adm"] = d.n_prior_adm.astype(int)
        dt, it, ot = (pd.to_datetime(d[c]) for c in ["adm_deathtime", "intime", "outtime"])
        o["icu_death_strict"] = (dt.notna() & (dt >= it) & (dt <= ot)).astype(int)   # death before ICU discharge
        # comorbidities from previous hospitalisations only (known before the index admission)
        for c in ["mi", "chf", "pvd", "dementia", "renal"]:
            o[f"prior_{c}"] = d[f"p_{c}"].astype(int)
        o["prior_liver_sev"] = d.p_severe_liver.astype(int)
        o["prior_liver_mild"] = ((d.p_mild_liver == 1) & (d.p_severe_liver == 0)).astype(int)
    else:
        acute = (d.acute_cvd == 1) | (d.tbi == 1) | (d.acute_brain_dx == 1)
        o["hemi_c"] = ((d.icd_hemi == 1) & ~acute).astype(int)
        o["cvd_c"] = ((d.ph_cvd == 1) | ((d.icd_cvd == 1) & (d.acute_cvd == 0))).astype(int)
        o["surg_dom"] = d.op_system.map(SURG_MAP_EICU).fillna("other")
        o["hospitalid"] = d.hospitalid
        o["apache_pred"] = pd.to_numeric(d.apache_pred_hosp, errors="coerce")
        # comorbidities from the structured past history only (recorded at ICU admission)
        for c in ["mi", "chf", "pvd", "dementia", "renal"]:
            o[f"prior_{c}"] = d[f"ph_{c}"].astype(int)
        o["prior_liver_sev"] = d.ph_severe_liver.astype(int)
        o["prior_liver_mild"] = ((d.ph_mild_liver == 1) & (d.ph_severe_liver == 0)).astype(int)
    # original Charlson Comorbidity Index (1987 weights, Quan definitions, without age points)
    o["cci"] = (d.mi + d.chf + d.pvd + d.cvd + d.dementia + d.cpd + d.rheum + d.pud
                + np.where(d.severe_liver == 1, 3, d.mild_liver) + np.where(d.dm_cc == 1, 2, d.dm_nocc)
                + 2 * d.hemi + 2 * d.renal + np.where(d.mets == 1, 6, 2 * d.malig_any) + 6 * d.aids)
    # same index, but with the harmonised (pre-existing) cerebrovascular disease and hemiplegia definitions
    o["cci_h"] = o.cci - d.cvd - 2 * d.hemi + o.cvd_c + 2 * o.hemi_c
    return o


AGE_DUM = ["age_50_59", "age_60_69", "age_70_79", "age_80p"]
COMORB = ["mi", "chf", "pvd", "cvd_c", "dementia", "cpd", "rheum", "pud", "liver_mild", "liver_sev",
          "dm", "renal", "solid_nonmet", "mets", "heme", "hemi_c"]
SURG_DUM = ["surg_neuro", "surg_thoracic", "surg_msk_trauma", "surg_other"]   # reference: abdominal/genitourinary


def add_dummies(o):
    o = o.copy()
    o["age_50_59"] = o.age.between(50, 59.99).astype(int)
    o["age_60_69"] = o.age.between(60, 69.99).astype(int)
    o["age_70_79"] = o.age.between(70, 79.99).astype(int)
    o["age_80p"] = (o.age >= 80).astype(int)
    for s in ["neuro", "thoracic", "msk_trauma", "other"]:
        o[f"surg_{s}"] = (o.surg_dom == s).astype(int)
    return o


mimic = add_dummies(harmonize(mimic_raw, "mimic"))
eicu = add_dummies(harmonize(eicu_raw, "eicu"))

# Prespecified models (no data-driven selection). Order matters for reproducibility of the bootstrap stream.
MODELS = {"M0_age_urgency": AGE_DUM + ["emergency"],
          "M1_unconstrained": AGE_DUM + ["emergency"] + COMORB,
          "M2_surgical_domain": AGE_DUM + ["emergency"] + COMORB + SURG_DUM,
          "mCCI-S": AGE_DUM + ["emergency"] + COMORB}
NONNEG = {"mCCI-S"}          # final score: negative points set to zero (confounding by indication)
FINAL = "mCCI-S"
OUTS = {}

## 3. Helper functions (DeLong AUC, calibration, decision curves, points model)

In [ ]:
def _midrank(x):
    j = np.argsort(x); z = x[j]; n = len(x); t = np.zeros(n); i = 0
    while i < n:
        k = i
        while k < n and z[k] == z[i]:
            k += 1
        t[i:k] = 0.5 * (i + k - 1) + 1
        i = k
    out = np.empty(n); out[j] = t
    return out


def delong(y, preds):
    y = np.asarray(y).astype(int); pos, neg = y == 1, y == 0
    m, n = pos.sum(), neg.sum()
    aucs, v01s, v10s = [], [], []
    for p in preds:
        p = np.asarray(p, float)
        tx, ty = _midrank(p[pos]), _midrank(p[neg])
        tz = _midrank(np.concatenate([p[pos], p[neg]]))
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        v01s.append((tz[:m] - tx) / n); v10s.append(1 - (tz[m:] - ty) / m)
    sx = np.atleast_2d(np.cov(np.vstack(v01s))); sy = np.atleast_2d(np.cov(np.vstack(v10s)))
    return np.array(aucs), sx / m + sy / n


def auc_ci(y, p):
    a, s = delong(y, [p]); se = np.sqrt(s[0, 0])
    return a[0], a[0] - 1.96 * se, a[0] + 1.96 * se


def auc_compare(y, p1, p2):
    a, s = delong(y, [p1, p2]); d = a[0] - a[1]
    se = np.sqrt(s[0, 0] + s[1, 1] - 2 * s[0, 1])
    return d, d - 1.96 * se, d + 1.96 * se, 2 * st.norm.sf(abs(d / se))


def calibration(y, p):
    y = np.asarray(y, float); p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); lp = logit(p)
    citl = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(), offset=lp).fit()
    slope = sm.GLM(y, sm.add_constant(lp, has_constant="add"), family=sm.families.Binomial()).fit()
    lo = sm.nonparametric.lowess(y, p, frac=2 / 3, it=0, return_sorted=False)
    ci_i, ci_s = citl.conf_int()[0], slope.conf_int()[1]
    return {"O": int(y.sum()), "E": float(p.sum()), "O_E": y.sum() / p.sum(),
            "citl": citl.params[0], "citl_lo": ci_i[0], "citl_hi": ci_i[1],
            "slope": slope.params[1], "slope_lo": ci_s[0], "slope_hi": ci_s[1],
            "brier": np.mean((p - y) ** 2), "brier_null": y.mean() * (1 - y.mean()),
            "brier_skill": 1 - np.mean((p - y) ** 2) / (y.mean() * (1 - y.mean())),
            "ICI": np.mean(np.abs(lo - p)), "E90": np.quantile(np.abs(lo - p), .9)}


def calib_deciles(y, p, label):
    q = pd.qcut(pd.Series(p).rank(method="first"), 10, labels=False)
    g = pd.DataFrame({"y": np.asarray(y), "p": np.asarray(p), "q": q}).groupby("q")
    r = g.agg(n=("y", "size"), events=("y", "sum"), predicted=("p", "mean"), observed=("y", "mean")).reset_index()
    r.insert(0, "label", label)
    return r


def dca(y, p, label, thr=np.arange(0.01, 0.31, 0.01)):
    y = np.asarray(y); n = len(y); prev = y.mean(); rows = []
    for t in thr:
        pos = p >= t
        nb = ((pos & (y == 1)).sum() - (pos & (y == 0)).sum() * t / (1 - t)) / n
        rows.append({"label": label, "threshold": round(t, 2), "net_benefit": nb,
                     "treat_all": prev - (1 - prev) * t / (1 - t)})
    return pd.DataFrame(rows)


def calib_slope(y, p):
    lp = logit(np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6))
    return sm.GLM(np.asarray(y, float), sm.add_constant(lp, has_constant="add"), family=sm.families.Binomial()).fit().params[1]


def fit_logit(df, cols, y="icu_death"):
    X = sm.add_constant(df[cols].astype(float), has_constant="add")
    return sm.GLM(df[y].astype(float), X, family=sm.families.Binomial()).fit()


def lp_from(beta, df, cols):
    return beta["const"] + df[cols].astype(float).values @ np.array([beta[c] for c in cols])


def build_points_model(df, cols, y="icu_death", nonneg=False):
    """Logistic model -> van Houwelingen uniform shrinkage -> integer points -> points-to-risk mapping."""
    m = fit_logit(df, cols, y)
    chi2 = m.null_deviance - m.deviance; k = len(cols)
    shrink = max(0.0, min(1.0, (chi2 - k) / chi2))
    b = m.params.copy(); b[cols] = b[cols] * shrink
    off = df[cols].astype(float).values @ b[cols].values
    b["const"] = sm.GLM(df[y].astype(float).values, np.ones((len(df), 1)), family=sm.families.Binomial(),
                        offset=off).fit().params[0]
    pts = {c: int(np.round(b[c] / POINT_UNIT)) for c in cols}
    if nonneg:   # clinical constraint: no predictor may lower risk -> negative points set to 0
        pts = {c: max(0, v) for c, v in pts.items()}
    score = df[cols].astype(int).values @ np.array([pts[c] for c in cols])
    mp = sm.GLM(df[y].astype(float).values, sm.add_constant(score.astype(float), has_constant="add"), family=sm.families.Binomial()).fit()
    return {"fit": m, "shrink": shrink, "beta_shrunk": b.to_dict(), "points": pts,
            "map": {"const": mp.params[0], "slope": mp.params[1]}, "cols": cols}


def score_of(model, df):
    return df[model["cols"]].astype(int).values @ np.array([model["points"][c] for c in model["cols"]])


def prob_of(model, df):
    return expit(model["map"]["const"] + model["map"]["slope"] * score_of(model, df))

## 4. Descriptive table

In [ ]:
def describe(o, name):
    rows = [("n", len(o), ""), ("age_median_IQR", f"{o.age.median():.0f} ({o.age.quantile(.25):.0f}-{o.age.quantile(.75):.0f})", "")]
    for c in ["emergency", "icu_death", "hosp_death"] + COMORB:
        rows.append((c, int(o[c].sum()), f"{100 * o[c].mean():.1f}"))
    for s in ["abdominal_gu", "neuro", "thoracic", "msk_trauma", "other"]:
        rows.append((f"surgery_{s}", int((o.surg_dom == s).sum()), f"{100 * (o.surg_dom == s).mean():.1f}"))
    return pd.DataFrame(rows, columns=["variable", f"{name}_n", f"{name}_pct"])


OUTS["T1_descriptive"] = describe(mimic, "MIMIC").merge(describe(eicu, "eICU"), on="variable")
OUTS["T1_descriptive"]

## 5. Development in MIMIC-IV and bootstrap internal validation
In every bootstrap resample the entire process (fit, shrinkage, rounding to points, constraint, points-to-risk
mapping) is repeated; optimism = mean(bootstrap performance - performance of the bootstrap model in the original data).

In [ ]:
dev = {}
coef_rows, iv_rows = [], []
for mname, cols in MODELS.items():
    pm = build_points_model(mimic, cols, nonneg=mname in NONNEG)
    dev[mname] = pm
    f = pm["fit"]; ci = f.conf_int()
    for c in ["const"] + cols:
        coef_rows.append({"model": mname, "variable": c, "beta": f.params[c], "OR": np.exp(f.params[c]),
                          "OR_lo": np.exp(ci.loc[c, 0]), "OR_hi": np.exp(ci.loc[c, 1]), "p": f.pvalues[c],
                          "beta_shrunk": pm["beta_shrunk"][c], "points": pm["points"].get(c, np.nan),
                          "n_present": int(mimic[c].sum()) if c != "const" else len(mimic),
                          "deaths_present": int(mimic.loc[mimic[c] == 1, "icu_death"].sum()) if c != "const" else int(mimic.icu_death.sum())})
    y = mimic.icu_death.values
    sc = score_of(pm, mimic); pr = prob_of(pm, mimic)
    lp_full = f.predict(sm.add_constant(mimic[cols].astype(float), has_constant="add"))
    app = {"AUC_LP": auc_ci(y, lp_full)[0], "AUC_points": auc_ci(y, sc)[0], "slope_points": calib_slope(y, pr),
           "brier_points": np.mean((pr - y) ** 2)}
    opt = {k: [] for k in app}
    for b in range(N_BOOT):
        idx = rng.integers(0, len(mimic), len(mimic)); bs = mimic.iloc[idx]
        try:
            pb = build_points_model(bs, cols, nonneg=mname in NONNEG)
        except Exception:
            continue
        yb = bs.icu_death.values
        fb = pb["fit"]
        lpb_b = fb.predict(sm.add_constant(bs[cols].astype(float), has_constant="add"))
        lpb_o = fb.predict(sm.add_constant(mimic[cols].astype(float), has_constant="add"))
        sb_b, sb_o = score_of(pb, bs), score_of(pb, mimic)
        pr_b, pr_o = prob_of(pb, bs), prob_of(pb, mimic)
        opt["AUC_LP"].append(auc_ci(yb, lpb_b)[0] - auc_ci(y, lpb_o)[0])
        opt["AUC_points"].append(auc_ci(yb, sb_b)[0] - auc_ci(y, sb_o)[0])
        opt["slope_points"].append(calib_slope(yb, pr_b) - calib_slope(y, pr_o))
        opt["brier_points"].append(np.mean((pr_b - yb) ** 2) - np.mean((pr_o - y) ** 2))
    for k, v in app.items():
        iv_rows.append({"model": mname, "measure": k, "apparent": v, "optimism": np.mean(opt[k]),
                        "corrected": v - np.mean(opt[k]), "n_bootstrap": len(opt[k])})
    print(mname, "shrinkage:", round(pm["shrink"], 3), "points:", pm["points"])

OUTS["T2_model_coefficients"] = pd.DataFrame(coef_rows)
OUTS["T3_internal_validation"] = pd.DataFrame(iv_rows)
pd.DataFrame(iv_rows).round(3)

## 6. Performance in MIMIC-IV (development) and eICU-CRD (external validation)

In [ ]:
fin = dev[FINAL]
disc_rows, cal_rows, dec_rows, dca_rows = [], [], [], []
for dname, o in [("MIMIC-IV_development", mimic), ("eICU-CRD_external", eicu)]:
    for yv in ["icu_death", "hosp_death"]:
        y = o[yv].values
        preds = {"mCCI-S": prob_of(fin, o)}
        for mname, pm in dev.items():
            if mname != FINAL:
                preds[mname] = prob_of(pm, o)
        preds["CCI"] = o.cci.values.astype(float)
        preds["CCI_harmonised"] = o.cci_h.values.astype(float)
        preds["emergency_alone"] = o.emergency.values.astype(float)
        for k, p in preds.items():
            a, lo, hi = auc_ci(y, p)
            r = {"dataset": dname, "outcome": yv, "model": k, "n": len(y), "deaths": int(y.sum()), "AUC": a, "AUC_lo": lo, "AUC_hi": hi}
            if k != "mCCI-S":
                d, dlo, dhi, pv = auc_compare(y, preds["mCCI-S"], p)
                r.update({"dAUC_mCCIS_minus_model": d, "d_lo": dlo, "d_hi": dhi, "p_DeLong": pv})
            disc_rows.append(r)
        phys = ("SOFA_day1", o.sofa.values, np.ones(len(o), bool)) if "sofa" in o else \
               ("APACHE_IVa_predicted", o.apache_pred.values, (o.apache_pred.notna() & (o.apache_pred > 0)).values)
        ok = phys[2]
        a, lo, hi = auc_ci(y[ok], phys[1][ok]); d, dlo, dhi, pv = auc_compare(y[ok], preds["mCCI-S"][ok], phys[1][ok])
        disc_rows.append({"dataset": dname, "outcome": yv, "model": phys[0], "n": int(ok.sum()), "deaths": int(y[ok].sum()),
                          "AUC": a, "AUC_lo": lo, "AUC_hi": hi, "dAUC_mCCIS_minus_model": d, "d_lo": dlo, "d_hi": dhi, "p_DeLong": pv})
        a, lo, hi = auc_ci(y[ok], preds["mCCI-S"][ok])        # mCCI-S in the same patients as the physiology score
        disc_rows.append({"dataset": dname, "outcome": yv, "model": f"mCCI-S_in_{phys[0]}_subset", "n": int(ok.sum()),
                          "deaths": int(y[ok].sum()), "AUC": a, "AUC_lo": lo, "AUC_hi": hi})
    y = o.icu_death.values
    for k in ["mCCI-S", "M0_age_urgency", "M1_unconstrained", "M2_surgical_domain"]:
        p = prob_of(dev[k], o)
        c = calibration(y, p); c.update({"dataset": dname, "model": k, "recalibration": "none"}); cal_rows.append(c)
        dec_rows.append(calib_deciles(y, p, f"{dname}|{k}|none"))
        lp = logit(np.clip(p, 1e-6, 1 - 1e-6))
        a0 = sm.GLM(y.astype(float), np.ones((len(y), 1)), family=sm.families.Binomial(), offset=lp).fit().params[0]
        c = calibration(y, expit(lp + a0)); c.update({"dataset": dname, "model": k, "recalibration": "intercept"}); cal_rows.append(c)
        dec_rows.append(calib_deciles(y, expit(lp + a0), f"{dname}|{k}|intercept"))
    for k in ["mCCI-S", "M0_age_urgency"]:
        dca_rows.append(dca(y, prob_of(dev[k], o), f"{dname}|{k}"))

OUTS["T4_discrimination"] = pd.DataFrame(disc_rows)
OUTS["T5_calibration"] = pd.DataFrame(cal_rows)
OUTS["T6_calibration_deciles"] = pd.concat(dec_rows)
OUTS["T7_decision_curves"] = pd.concat(dca_rows)
pd.DataFrame(cal_rows)[["dataset", "model", "recalibration", "O_E", "citl", "slope", "ICI", "brier"]].round(3)

## 7. Risk groups

In [ ]:
grp = []
for dname, o in [("MIMIC-IV_development", mimic), ("eICU-CRD_external", eicu)]:
    s = score_of(fin, o); p = prob_of(fin, o)
    g = pd.cut(s, [-1, 2, 5, 8, 11, 99], labels=["0-2", "3-5", "6-8", "9-11", ">=12"])
    t = pd.DataFrame({"g": g, "y": o.icu_death.values, "yh": o.hosp_death.values, "p": p, "s": s})
    for lab, x in t.groupby("g", observed=False):
        grp.append({"dataset": dname, "group": lab, "n": len(x), "icu_deaths": int(x.y.sum()), "hosp_deaths": int(x.yh.sum()),
                    "mean_predicted": x.p.mean()})
    grp.append({"dataset": dname, "group": "median_IQR", "n": float(np.median(s)), "icu_deaths": float(np.percentile(s, 25)),
                "hosp_deaths": float(np.percentile(s, 75)), "mean_predicted": np.nan})
OUTS["T8_risk_groups"] = pd.DataFrame(grp)
pts_rows = []        # calibration by score value (no arbitrary splitting of tied scores)
for dname, o in [("MIMIC-IV_development", mimic), ("eICU-CRD_external", eicu)]:
    t = pd.DataFrame({"s": score_of(fin, o), "y": o.icu_death.values, "p": prob_of(fin, o)})
    for s, x in t.groupby("s"):
        pts_rows.append({"dataset": dname, "points": int(s), "n": len(x), "icu_deaths": int(x.y.sum()), "predicted": x.p.mean()})
OUTS["T8b_calibration_by_points"] = pd.DataFrame(pts_rows)
OUTS["T8_risk_groups"]

## 8. Incremental value over physiology-based scores (tenfold cross-validation)
MIMIC-IV (development data): first-day SOFA; the **entire mCCI-S construction** (model, shrinkage, points,
constraint) is repeated within each training fold, so the held-out fold never contributes to the score
(nested cross-validation). eICU-CRD (external data): the fixed MIMIC-IV score is added to the logit of APACHE IVa
predicted hospital mortality; only the combination weights are cross-validated. Fold assignment uses a fixed seed
per analysis so that results do not depend on execution order.

In [ ]:
def cv_pred(df, phys, y, seed, k=10, score="none"):
    """score: 'none' = physiology only; 'fixed' = precomputed mccis_points; 'nested' = score rebuilt in each training fold."""
    folds = np.random.default_rng(seed).integers(0, k, len(df)); pred = np.zeros(len(df))
    for f in range(k):
        tr, te = folds != f, folds == f
        dtr, dte = df[tr].copy(), df[te].copy()
        cols = [phys]
        if score == "fixed":
            cols = [phys, "mccis_points"]
        elif score == "nested":
            pm = build_points_model(dtr, MODELS[FINAL], nonneg=True)          # always developed for ICU death
            dtr["sc_cv"], dte["sc_cv"] = score_of(pm, dtr), score_of(pm, dte); cols = [phys, "sc_cv"]
        m = fit_logit(dtr, cols, y)
        pred[te] = m.predict(sm.add_constant(dte[cols].astype(float), has_constant="add"))
    return pred


inc_rows = []
mimic["mccis_points"] = score_of(fin, mimic); eicu["mccis_points"] = score_of(fin, eicu)
eicu_ap = eicu[eicu.apache_pred.notna() & (eicu.apache_pred > 0)].copy()
eicu_ap["apache_logit"] = logit(eicu_ap.apache_pred.clip(1e-4, 1 - 1e-4))
for i, (dname, o, phys, mode) in enumerate([("MIMIC-IV", mimic, "sofa", "nested"), ("MIMIC-IV", mimic, "sofa", "fixed"),
                                             ("eICU-CRD", eicu_ap, "apache_logit", "fixed")]):
    for j, yv in enumerate(["icu_death", "hosp_death"]):
        y = o[yv].values; seed = SEED + 10 * (0 if dname == "MIMIC-IV" else 1) + j
        p_phys = cv_pred(o, phys, yv, seed); p_both = cv_pred(o, phys, yv, seed, score=mode)
        d, dlo, dhi, pv = auc_compare(y, p_both, p_phys)
        lr = fit_logit(o, [phys], yv).deviance - fit_logit(o, [phys, "mccis_points"], yv).deviance
        inc_rows.append({"dataset": dname, "outcome": yv, "physiology": phys, "score_in_cv": mode, "n": len(y),
                         "deaths": int(y.sum()), "AUC_physiology": auc_ci(y, p_phys)[0], "AUC_combined": auc_ci(y, p_both)[0],
                         "dAUC": d, "d_lo": dlo, "d_hi": dhi, "p_DeLong": pv, "LR_chi2": lr, "p_LR": st.chi2.sf(lr, 1)})
OUTS["T9_incremental_value"] = pd.DataFrame(inc_rows)
pd.DataFrame(inc_rows).round(4)


## 9. Between-hospital heterogeneity in eICU-CRD (DerSimonian-Laird random effects)
AUC: hospitals with >=5 deaths and >=5 survivors. O/E: hospitals with >=100 patients (variance approximated by 1/E).

In [ ]:
def dl_meta(est, se):
    w = 1 / se ** 2; fe = np.sum(w * est) / np.sum(w); q = np.sum(w * (est - fe) ** 2); df = len(est) - 1
    tau2 = max(0, (q - df) / (np.sum(w) - np.sum(w ** 2) / np.sum(w)))
    wr = 1 / (se ** 2 + tau2); re = np.sum(wr * est) / np.sum(wr); se_re = np.sqrt(1 / np.sum(wr))
    i2 = max(0, (q - df) / q) if q > 0 else 0
    tcrit = st.t.ppf(.975, max(df - 1, 1)); pi = np.sqrt(tau2 + se_re ** 2) * tcrit
    return re, se_re, tau2, i2, re - pi, re + pi


hosp_rows = []
eicu["_p"] = prob_of(fin, eicu)
for hid, g in eicu.groupby("hospitalid"):
    ev, ne = g.icu_death.sum(), (1 - g.icu_death).sum()
    r = {"hospital": hid, "n": len(g), "deaths": int(ev), "E": g._p.sum(), "AUC": np.nan, "AUC_se": np.nan}
    if ev >= 5 and ne >= 5:
        a, s = delong(g.icu_death.values, [g._p.values]); r["AUC"], r["AUC_se"] = a[0], np.sqrt(s[0, 0])
    if ev >= 5 or len(g) >= 100:
        hosp_rows.append(r)
Hh = pd.DataFrame(hosp_rows)
Hh["O_E"] = np.where(Hh.deaths > 0, Hh.deaths, 0.5) / Hh.E
meta = {}
Ha = Hh[Hh.AUC.notna()]
if len(Ha) >= 3:
    lg = logit(Ha.AUC.clip(.501, .999)); lg_se = Ha.AUC_se / (Ha.AUC * (1 - Ha.AUC))
    re, se_re, tau2, i2, plo, phi = dl_meta(lg.values, lg_se.values)
    meta["AUC"] = {"pooled": expit(re), "lo": expit(re - 1.96 * se_re), "hi": expit(re + 1.96 * se_re),
                   "PI_lo": expit(plo), "PI_hi": expit(phi), "I2": i2, "tau2": tau2, "k": len(Ha)}
Ho = Hh[Hh.n >= 100]
if len(Ho) >= 3:
    re, se_re, tau2, i2, plo, phi = dl_meta(np.log(Ho.O_E).values, (1 / np.sqrt(Ho.E)).values)
    meta["O_E"] = {"pooled": np.exp(re), "lo": np.exp(re - 1.96 * se_re), "hi": np.exp(re + 1.96 * se_re),
                   "PI_lo": np.exp(plo), "PI_hi": np.exp(phi), "I2": i2, "tau2": tau2, "k": len(Ho)}
    Hz = Ho[Ho.deaths > 0]          # sensitivity: variance of log(O/E) approximated by 1/O (hospitals with >=1 death)
    re, se_re, tau2, i2, plo, phi = dl_meta(np.log(Hz.deaths / Hz.E).values, (1 / np.sqrt(Hz.deaths)).values)
    meta["O_E_var_1_over_O"] = {"pooled": np.exp(re), "lo": np.exp(re - 1.96 * se_re), "hi": np.exp(re + 1.96 * se_re),
                                "PI_lo": np.exp(plo), "PI_hi": np.exp(phi), "I2": i2, "tau2": tau2, "k": len(Hz),
                                "zero_death_hospitals_excluded": int((Ho.deaths == 0).sum())}
if "AUC" in meta:
    meta["AUC"].update({"patients_in_k": int(Ha.n.sum()), "deaths_in_k": int(Ha.deaths.sum()),
                        "hospitals_total": int(eicu.hospitalid.nunique()), "patients_total": len(eicu),
                        "deaths_total": int(eicu.icu_death.sum())})
Hh = Hh.sort_values("AUC").reset_index(drop=True)
Hh["hospital"] = [f"H{i + 1:02d}" for i in range(len(Hh))]       # anonymised index
OUTS["T10_hospitals"] = Hh
OUTS["T11_meta_analysis"] = pd.DataFrame(meta).T.reset_index().rename(columns={"index": "measure"})
OUTS["T11_meta_analysis"]

## 10. Sensitivity analyses (fixed mCCI-S, no refitting)
1. MIMIC-IV ICU death restricted to deaths before ICU discharge (primary definition also counts deaths within 6 h
   after ICU discharge).
2. Emergency surgery restricted to emergency/urgent admission types (MIMIC-IV).
3. Comorbidities known before the index admission only (MIMIC-IV: previous hospitalisations; eICU-CRD:
   structured past history).


In [ ]:
SCORE_COMORB = {"mi": "prior_mi", "chf": "prior_chf", "pvd": "prior_pvd", "dementia": "prior_dementia",
                "renal": "prior_renal", "liver_sev": "prior_liver_sev", "liver_mild": "prior_liver_mild"}
sens = []


def sens_row(analysis, dname, y, p, ref=None, **extra):
    a, lo, hi = auc_ci(y, p)
    r = {"analysis": analysis, "dataset": dname, "n": len(y), "icu_deaths": int(np.sum(y)), "AUC": a, "AUC_lo": lo,
         "AUC_hi": hi, "O_E": np.sum(y) / np.sum(p)}
    if ref is not None:
        d, dlo, dhi, pv = auc_compare(y, ref, p); r.update({"dAUC_primary_minus_this": d, "d_lo": dlo, "d_hi": dhi, "p_DeLong": pv})
    r.update(extra); sens.append(r)


for dname, o in [("MIMIC-IV", mimic), ("eICU-CRD", eicu)]:
    y = o.icu_death.values; p0 = prob_of(fin, o)
    sens_row("primary", dname, y, p0)
    q = o.copy()
    for c, pc in SCORE_COMORB.items():
        q[c] = q[pc]
    sens_row("comorbidities_known_before_admission", dname, y, prob_of(fin, q), ref=p0,
             patients_with_any_score_comorbidity_primary=int((o[list(SCORE_COMORB)].sum(axis=1) > 0).sum()),
             patients_with_any_score_comorbidity_prior=int((o[list(SCORE_COMORB.values())].sum(axis=1) > 0).sum()))
    if dname == "MIMIC-IV":
        ys = o.icu_death_strict.values
        sens_row("icu_death_before_discharge_only", dname, ys, p0,
                 deaths_reclassified=int(((o.icu_death == 1) & (o.icu_death_strict == 0)).sum()))
        q = o.copy(); q["emergency"] = q["emerg_strict"]
        sens_row("emergency_strict_admission_types", dname, y, prob_of(fin, q), ref=p0,
                 n_emergency_primary=int(o.emergency.sum()), n_emergency_strict=int(o.emerg_strict.sum()))
        sub = o.n_prior_adm > 0
        q = o[sub].copy()
        for c, pc in SCORE_COMORB.items():
            q[c] = q[pc]
        sens_row("subset_with_previous_admission_primary", dname, o.icu_death.values[sub], p0[sub.values])
        sens_row("subset_with_previous_admission_prior_only", dname, o.icu_death.values[sub], prob_of(fin, q), ref=p0[sub.values])
OUTS["T12_sensitivity"] = pd.DataFrame(sens)
OUTS["T12_sensitivity"].round(3)


## 11. Export (aggregate results only)

In [ ]:
spec = {"POINT_UNIT": POINT_UNIT, "N_BOOT": N_BOOT, "SYNTHETIC": SYNTHETIC, "SEED": SEED,
        "models": {k: {"points": v["points"], "map": v["map"], "shrink": v["shrink"], "beta_shrunk": v["beta_shrunk"],
                       "cols": v["cols"]} for k, v in dev.items()},
        "meta": meta, "n_mimic": len(mimic), "n_eicu": len(eicu)}
import sys, scipy, statsmodels, matplotlib as _mpl
spec["software"] = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
                    "scipy": scipy.__version__, "statsmodels": statsmodels.__version__, "matplotlib": _mpl.__version__}
json.dump(spec, open(f"{OUTDIR}/model_spec.json", "w"), indent=2, default=float)
for k, v in OUTS.items():
    v.to_csv(f"{OUTDIR}/{k}.csv", index=False)
with zipfile.ZipFile("mccis_results.zip", "w") as z:
    for fn in os.listdir(OUTDIR):
        z.write(os.path.join(OUTDIR, fn), fn)
print("Done: mccis_results.zip ->", sorted(os.listdir(OUTDIR)))
if not SYNTHETIC:
    from google.colab import files
    files.download("mccis_results.zip")